In [22]:
"""
Notebook: 01_data_collection.ipynb

Purpose:
Load and inspect idiom datasets collected from multiple sources.

Dataset location:
../data/

Author: Ayman
Project: IdiomX
"""

'\nNotebook: 01_data_collection.ipynb\n\nPurpose:\nLoad and inspect idiom datasets collected from multiple sources.\n\nDataset location:\n../data/\n\nAuthor: Ayman\nProject: IdiomX\n'

In [23]:
from pathlib import Path
import sys

# Project directories
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
DATA_RAW_DIR = DATA_DIR / "raw"
DATA_PROCESS_DIR = DATA_DIR / "processed"

# python file path 
SCRIPTS_DIR = BASE_DIR / "scripts"

# Make the scripts folder importable
sys.path.append(str(SCRIPTS_DIR))

# Make sure directories exist
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# download the kaikki.org-dictionary-English-words
import requests

file_path = DATA_RAW_DIR / "kaikki.org-dictionary-English-words.jsonl"

if not file_path.exists():
    print("Downloading Kaikki dataset...")
    # External dataset sources
    url = "https://kaikki.org/dictionary/English/words/kaikki.org-dictionary-English-words.jsonl"
    
    r = requests.get(url, stream=True)

    with open(file_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Download complete")
else:
    print("Dataset already exists")

Dataset already exists


## 1. Extract Idioms from Wiktionary (Kaikki Dataset)

Define the input and output paths used in the idiom extraction pipeline.  
The raw Kaikki Wiktionary dataset is specified as the primary input source.  

In [64]:
# CONFIG input and output dataset 

# Raw dataset file
KAIKKI_FILE = DATA_RAW_DIR / "kaikki.org-dictionary-English-words.jsonl"

# External dataset sources if required
KAIKKI_DATASET_URL = "https://kaikki.org/dictionary/English/words/kaikki.org-dictionary-English-words.jsonl"

# output extracted files
KAIKKI_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki.csv"
KAIKKI_JSONL = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki.jsonl"

In [5]:
# import the function from scripts/ extraction python file
from extract_idioms_from_kaikki import extract_kaikki_idioms

In [12]:
summary = extract_kaikki_idioms(
    input_file=KAIKKI_FILE,
    output_csv=KAIKKI_CSV,
    output_jsonl=KAIKKI_JSONL,
    strict_mode=False,
)

summary

Reading Kaikki JSONL: 1454988it [02:43, 8893.37it/s] 



Extraction finished.
{'input_file': '..\\data\\raw\\kaikki.org-dictionary-English-words.jsonl', 'output_csv': '..\\data\\processed\\idioms_wiktionary_kaikki.csv', 'output_jsonl': '..\\data\\processed\\idioms_wiktionary_kaikki.jsonl', 'strict_mode': False, 'total_lines_scanned': 1454988, 'rows_kept': 229807}


{'input_file': '..\\data\\raw\\kaikki.org-dictionary-English-words.jsonl',
 'output_csv': '..\\data\\processed\\idioms_wiktionary_kaikki.csv',
 'output_jsonl': '..\\data\\processed\\idioms_wiktionary_kaikki.jsonl',
 'strict_mode': False,
 'total_lines_scanned': 1454988,
 'rows_kept': 229807}

In [4]:
import pandas as pd

df_kaikki = pd.read_csv(KAIKKI_CSV, encoding="utf-8-sig")
print("dataset shape: ",df_kaikki.shape)
df_kaikki.head()

dataset shape:  (229807, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,rain cats and dogs,To rain very heavily.,"[""]Those weather-gaws aren't out for nothing. ...",verb,"idiomatic, impersonal",1,kaikki_wiktionary,0
1,Pope Julius,A sixteenth-century gambling card game about w...,Of Pope Julius cardys he ys chefe cardynall.,name,obsolete,0,kaikki_wiktionary,0
2,GNU FDL,partial Initialism of GNU Free Documentation L...,NaN,name,NaN,0,kaikki_wiktionary,0
3,current events,Current affairs; those events and issues of in...,The teacher asked the students to write a repo...,noun,"plural, plural-normally",0,kaikki_wiktionary,0
4,false friend,A word in a language that bears a deceptive re...,A word and its false friend may well be etymol...,noun,NaN,0,kaikki_wiktionary,0


In [5]:
print("Rows:", len(df_kaikki))
print("Unique idioms:", df_kaikki["idiom"].nunique())
print("\nPOS distribution:")
print(df_kaikki["pos"].value_counts().head(10))

Rows: 229807
Unique idioms: 193390

POS distribution:
pos
noun           127869
verb            51656
name            33492
adj              4508
phrase           3486
prep_phrase      3484
adv              1947
intj             1612
proverb           943
prep              331
Name: count, dtype: int64


## 2. Strict Filtering of Extracted Wiktionary Idioms

The initial extraction from the Kaikki Wiktionary dataset includes many lexical entries that are not strictly idiomatic expressions.  
To improve dataset quality, a strict filtering stage is applied to retain only strong idiom candidates.

This filtering step removes entries that are likely to represent literal phrases, incomplete expressions, or non-idiomatic multi-word constructions. The filtered output produces a cleaner subset of idioms that can later be merged with other idiom datasets.

In [6]:
from collect_02_filter_strict_idioms import filter_strict_idioms

In [7]:
KAIKKI_STRICT_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_strict.csv"
KAIKKI_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki.csv"

df_kaikki_strict = filter_strict_idioms(KAIKKI_CSV,KAIKKI_STRICT_CSV )

print(df_kaikki_strict.shape)
df_kaikki_strict.head()

Saved: ..\data\processed\idioms_wiktionary_kaikki_strict.csv
Rows: 30749
(30749, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,$100 hamburger,A general aviation flight that involves flying...,It’s called hunting the $100 hamburger — “$100...,noun,slang,0,kaikki_wiktionary,0
1,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,noun,,1,kaikki_wiktionary,1
2,& ux.,Obsolete form of et ux..,,phrase,"alt-of, obsolete",0,kaikki_wiktionary,0
3,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,phrase,informal,0,kaikki_wiktionary,0
4,'ark at 'ee,Used to draw attention to something or someone.,Then a lady came into the shop and saw the T-s...,phrase,informal,0,kaikki_wiktionary,1


In [8]:
before_count = len(df_kaikki)
after_count = len(df_kaikki_strict)

print("Before strict filtering:", before_count)
print("After strict filtering:", after_count)
print("Retention rate:", round(after_count / before_count * 100, 2), "%")
print("Unique idioms:", df_kaikki_strict["idiom"].nunique())

df_kaikki_strict.sample(10, random_state=42)

Before strict filtering: 229807
After strict filtering: 30749
Retention rate: 13.38 %
Unique idioms: 24938


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
19795,one nail drives out another,"New people, things, or customs eventually supp...",,proverb,,0,kaikki_wiktionary,0
21000,play away,To be sexually unfaithful out of one's home,,verb,"intransitive, slang",0,kaikki_wiktionary,0
10493,follow the crowd,"To conform to majority beliefs, opinions, or p...","As much as the next guy, I don't like to follo...",verb,idiomatic,1,kaikki_wiktionary,0
10919,fuck this for a game of soldiers,Synonym of sod this for a game of soldiers.,"Fuck this for a game of soldiers, I'm gonna ni...",phrase,vulgar,0,kaikki_wiktionary,0
7363,crap circus,A thoroughly chaotic and disorganized situatio...,I didn't have a scanner with me but it certain...,noun,"slang, vulgar",0,kaikki_wiktionary,0
9434,emperor's new clothes,Something obvious and embarrassing that is pol...,Marks & Spencer announced a 19% fall in annual...,noun,"idiomatic, plural, plural-only",1,kaikki_wiktionary,0
15181,jail bars,Used other than figuratively or idiomatically:...,,noun,"plural, plural-only",1,kaikki_wiktionary,1
20202,pa chew cheng,Of a man: to masturbate.,"2001, Anonymous, Worst Job in Singapore, Googl...",verb,"Singapore, colloquial",0,kaikki_wiktionary,0
3298,be cruel to be kind,To act cruelly in order to achieve a positive ...,"I muſt be cruell only to be kinde, / This bad ...",verb,idiomatic,1,kaikki_wiktionary,0
2276,anal vore,To insert a character into the rectum.,"In the comic, the giant predator anal vores it...",verb,"informal, slang",0,kaikki_wiktionary,0


## 3. Cleaning the Strict Idiom Dataset

An additional cleaning stage is performed to normalize and refine the extracted idioms.

By standardizes text fields, removes residual noise, and ensures that idioms and their associated metadata follow a consistent format.

In [9]:
# clean idioms
from collect_03_clean_idioms import clean_idioms

In [10]:
KAIKKI_STRICT_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_strict.csv"
KAIKKI_CLEANED_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_cleaned.csv"

df_kaikki_clean = clean_idioms(KAIKKI_STRICT_CSV,KAIKKI_CLEANED_CSV )

print(df_kaikki_clean.shape)
df_kaikki_clean.head()

Saved: ..\data\processed\idioms_wiktionary_kaikki_cleaned.csv
Rows: 21816
(21816, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,$100 hamburger,A general aviation flight that involves flying...,It’s called hunting the $100 hamburger — “$100...,noun,slang,0,kaikki_wiktionary,0
1,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,phrase,informal,0,kaikki_wiktionary,0
2,'d best,Had best.,It's getting late. You'd best get on home.,verb,"auxiliary, colloquial, modal",0,kaikki_wiktionary,0
3,'er indoors,A person's wife.,,pron,"UK, slang",0,kaikki_wiktionary,0
4,'fraid so,I am afraid so,,phrase,slang,0,kaikki_wiktionary,0


## 4. Constructing a High-Precision Idiom Subset

Identify idioms with stronger evidence of figurative usage.  

By applying additional heuristics and linguistic indicators to prioritize expressions that are more likely to represent true idiomatic constructions. These include indicators such as idiom-related tags, semantic descriptions suggesting figurative meaning, and patterns commonly associated with idiomatic expressions.

In [11]:
# building high precision idioms
from collect_04_build_high_precision_idioms import high_precision_idioms

In [12]:
KAIKKI_CLEANED_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_cleaned.csv"
KAIKKI_HIGH_PRECISION_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_high_precision.csv"

df_kaikki_high_precision = high_precision_idioms(KAIKKI_CLEANED_CSV,KAIKKI_HIGH_PRECISION_CSV )

print(df_kaikki_high_precision.shape)
df_kaikki_high_precision.head()

Saved: ..\data\processed\idioms_wiktionary_kaikki_high_precision.csv
Rows: 13202
(13202, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,phrase,informal,0,kaikki_wiktionary,0
1,'fraid so,I am afraid so,,phrase,slang,0,kaikki_wiktionary,0
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",phrase,informal,0,kaikki_wiktionary,0
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",phrase,,0,kaikki_wiktionary,0
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",adj,"idiomatic, not-comparable",1,kaikki_wiktionary,0


## 5. Normalizing High-Precision Wiktionary Idioms

A normalization step is applied to standardize the structure and textual representation of the entries.

To ensures that idioms, meanings, examples, and metadata follow a consistent schema across all records. 

Text normalization includes:

    - Standardizing formatting, 
    - Removing irregular whitespace, 
    - Aligning column structures 
    
so the dataset can be easily merged with idioms collected from other sources.

In [13]:
# building high precision normalized idioms
from collect_05_normalize_kaikki_high_precision import normalize_high_precision_idioms

In [14]:
KAIKKI_HIGH_PRECISION_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_high_precision.csv"
KAIKKI_NORMALIZED_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_normalized.csv"

df_kaikki_normalized = normalize_high_precision_idioms(KAIKKI_HIGH_PRECISION_CSV,KAIKKI_NORMALIZED_CSV )

print(df_kaikki_normalized.shape)
df_kaikki_normalized.head()

Saved: ..\data\processed\idioms_source_kaikki_normalized.csv
Rows: 13202
(13202, 8)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high
1,'fraid so,I am afraid so,,kaikki_wiktionary,dictionary,phrase,slang,high
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",kaikki_wiktionary,dictionary,phrase,,high
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",kaikki_wiktionary,dictionary,adj,"idiomatic, not-comparable",high


## 6. Extracting Multi-Word Expressions from WordNet

To expand the lexical coverage of idiomatic expressions, multi-word expressions are extracted from the WordNet lexical database. WordNet is a widely used lexical resource that organizes English words into synonym sets (synsets) and provides semantic relationships and definitions.

In [65]:
from collect_09_extract_wordnet_multiword_expressions import extract_wordnet_multiword_expressions

WORDNET_CSV = DATA_PROCESS_DIR / "idioms_source_wordnet_normalized.csv"

df_wordnet = extract_wordnet_multiword_expressions(WORDNET_CSV)

print(df_wordnet.shape)
df_wordnet.head()

Saved: ..\data\processed\idioms_source_wordnet_normalized.csv
Rows: 68072
Unique idioms: 64243
(68072, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,ready to hand,easy to reach,,wordnet,lexical_database,s,,medium,
1,out of reach,inaccessibly located or situated,,wordnet,lexical_database,s,,medium,
2,dead on target,accurately placed or thrown,,wordnet,lexical_database,s,,medium,
3,wide of the mark,not on target,,wordnet,lexical_database,s,,medium,
4,used to,in the habit; ; ; - Henry David Thoreau,,wordnet,lexical_database,s,,medium,


### Dataset Expansion with WordNet

After integrating WordNet multi-word expressions, the dataset contains more than 68,000 candidate expressions from WordNet alone. While many of these expressions are not strictly idiomatic, they significantly expand the lexical coverage of the dataset and will later be refined through global idiom filtering.

## 7. Merging WordNet Expressions with the Main Idiom Dataset

WordNet expressions are labeled with a medium idiom confidence level because they include both idiomatic and compositional multi-word expressions.

The candidate multi-word expressions extracted from WordNet are merged with the existing idiom dataset already contains curated idioms from Wiktionary.

In [16]:
from collect_10_merge_wordnet_with_stage3 import merge_stage3_with_wordnet

STAGE3_CSV = DATA_PROCESS_DIR / "idioms_source_kaikki_normalized.csv"
WORDNET_CSV = DATA_PROCESS_DIR / "idioms_source_wordnet_normalized.csv"
STAGE4_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage.csv"

df_stage4 = merge_stage3_with_wordnet(
    stage3_file=STAGE3_CSV,
    wordnet_file=WORDNET_CSV,
    output_file=STAGE4_CSV
)

print(df_stage4.shape)
df_stage4.head()

Saved: ..\data\processed\idioms_dataset_stage.csv
Rows: 81274
Unique idioms: 75379

Source distribution:
source
wordnet              68072
kaikki_wiktionary    13202
Name: count, dtype: int64
(81274, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high,
1,'fraid so,I am afraid so,,kaikki_wiktionary,dictionary,phrase,slang,high,
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high,
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",kaikki_wiktionary,dictionary,phrase,,high,
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",kaikki_wiktionary,dictionary,adj,"idiomatic, not-comparable",high,


### Dataset Expansion after WordNet Integration

After integrating WordNet multi-word expressions, the dataset expanded significantly to more than 80,000 expressions. 

the dataset at this stage contains both idiomatic and non-idiomatic phrases. 

These expressions will be refined in the subsequent global idiom filtering stage to produce the final high-quality idiom dataset.

data/processed/

- idioms_dataset_stage.csv ->#  merged raw sources

- idioms_dataset_stage_broad.csv ->#  large dataset For training an LLM or large model

- idioms_dataset_stage5_high_precision.csv ->#  clean dataset for research

### IdiomX-Core

High quality idioms

    idioms_dataset_stage_high_precision.csv

### IdiomX-Extended

Broader idiomatic expressions

    idioms_dataset_stage_broad.csv

In [27]:
try:
    from rdflib import Graph, Literal
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdflib"])
    from rdflib import Graph, Literal
try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

## 8. Global Idiom Filtering

Now dataset contains a large number of expressions that are not strictly idiomatic. In particular, WordNet contributes many compositional multi-word expressions that must be filtered to ensure dataset quality.

A global idiom filtering procedure is applied to the entire merged dataset. The filtering process significantly reduces the dataset size while improving idiom quality and reliability.

In [17]:
from collect_12_01_filter_global_idioms import filter_global_idioms

STAGE4_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage.csv"
STAGE5_BROAD_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage_broad.csv"
STAGE5_HIGH_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage5_high_precision.csv"

df_broad, df_high, stats = filter_global_idioms(
    input_file=STAGE4_CSV,
    output_broad=STAGE5_BROAD_CSV,
    output_high=STAGE5_HIGH_CSV
)

print(stats)
print(df_broad.shape, df_high.shape)

df_broad.head()
df_high.head()

C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\notebooks\..\scripts\collect_12_01_filter_global_idioms.py:317: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, encoding="utf-8-sig")


Saved broad: ..\data\processed\idioms_dataset_stage_broad.csv
Saved high : ..\data\processed\idioms_dataset_stage5_high_precision.csv
Rows after basic cleaning: 78598
Rows broad: 15344
Rows high precision: 12853
Unique idioms broad: 15082
Unique idioms high precision: 12847

Broad per source:
source
kaikki_wiktionary    12838
wordnet               2506
Name: count, dtype: int64

High per source:
source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64
{'rows_input': 78598, 'rows_broad': 15344, 'rows_high_precision': 12853, 'unique_idioms_input': 72903, 'unique_idioms_broad': 15082, 'unique_idioms_high_precision': 12847, 'broad_source_counts': {'kaikki_wiktionary': 12838, 'wordnet': 2506}, 'high_source_counts': {'kaikki_wiktionary': 12838, 'wordnet': 15}}
(15344, 9) (12853, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high,
1,'fraid so,I am afraid so,,kaikki_wiktionary,dictionary,phrase,slang,high,
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high,
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",kaikki_wiktionary,dictionary,phrase,,high,
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",kaikki_wiktionary,dictionary,adj,"idiomatic, not-comparable",high,


### Dataset Quality Improvement after Global Filtering

The global idiom filtering stage reduces the dataset from more than 75,000 candidate expressions to approximately 13,000 high-precision idioms. This filtering process removes a large number of compositional phrases originating primarily from WordNet, significantly improving the linguistic quality of the dataset.

The resulting high-precision idiom dataset represents the final curated idiom collection used in the IdiomX project.

#### Broad Dataset (larger coverage)

idioms_dataset_stage_broad.csv

Contains:

- idioms

- idiom-like expressions

- some multi-word expressions

- lexical phrases from WordNet

- looser filtering

This dataset is bigger but slightly noisier.

Example types included:

    break the ice
    kick the bucket
    take into account
    state of the art
    in the long run

##### Good for:

- training large language models

- semantic search

- embedding training

- exploratory analysis

#### High-Precision Dataset (very clean)

idioms_dataset_stage_high_precision.csv

Contains only strong idioms, based on:

- idiom tags

- phrase dictionaries

- idiom confidence

- high-quality sources

Example:

    break the ice
    kick the bucket
    spill the beans
    bite the bullet

#### Good for:

- academic dataset release

- idiom translation model

- idiom detection model

- benchmark dataset

## 9. Final Dataset Statistics

In [19]:
from collect_12_02_dataset_statistics import build_dataset_statistics

STAGE6_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage5_high_precision.csv"
STATS_JSON = DATA_PROCESS_DIR / "idioms_dataset_stage_statistics.json"
STATS_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage_source_distribution.csv"

stats, df_source_stats = build_dataset_statistics(
    input_file=STAGE6_CSV,
    output_json=STATS_JSON,
    output_csv=STATS_CSV
)

print(stats)
df_source_stats.head()

Saved JSON stats: ..\data\processed\idioms_dataset_stage_statistics.json
Saved source distribution CSV: ..\data\processed\idioms_dataset_stage_source_distribution.csv

Summary:
Rows total: 12853
Unique idioms: 12847
Rows with meaning: 12853
Rows with example: 8891
Average idiom tokens: 3.23

Source distribution:
source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64
{'rows_total': 12853, 'unique_idioms': 12847, 'unique_meanings': 12525, 'rows_with_meaning': 12853, 'rows_with_example': 8891, 'rows_without_meaning': 0, 'rows_without_example': 3962, 'avg_idiom_tokens': 3.23, 'min_idiom_tokens': 2, 'max_idiom_tokens': 7, 'source_distribution': {'kaikki_wiktionary': 12838, 'wordnet': 15}, 'source_type_distribution': {'dictionary': 12838, 'lexical_database': 15}, 'idiom_confidence_distribution': {'high': 12838, 'medium': 15}}


,source,count
0,kaikki_wiktionary,12838
1,wordnet,15


## Final Dataset Summary

The IdiomX dataset representing **12,853 unique idiomatic expressions** collected from multiple lexical resources.It provides a large-scale resource for studying idiomatic language, semantic interpretation, and idiom translation tasks.

In [24]:
from collect_12_02_dataset_statistics import build_dataset_statistics
from pathlib import Path
import shutil

# Input dataset (processed)
# ----------------------------------------------------
STAGE6_CSV = DATA_PROCESS_DIR / "idioms_dataset_stage5_high_precision.csv"

# Statistics output (processed)
# ----------------------------------------------------
STATS_JSON = DATA_PROCESS_DIR / "idiomx_dataset_v1_statistics.json"
STATS_CSV = DATA_PROCESS_DIR / "idiomx_dataset_v1_source_distribution.csv"

stats, df_source_stats = build_dataset_statistics(
    input_file=STAGE6_CSV,
    output_json=STATS_JSON,
    output_csv=STATS_CSV
)

print(stats)
display(df_source_stats.head())


# Save FINAL dataset copies
# ====================================================

DATA_FINAL_DIR = BASE_DIR / "data" / "final"
DATA_FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET = DATA_FINAL_DIR / "idiomx_dataset_v1.csv"
FINAL_STATS_JSON = DATA_FINAL_DIR / "idiomx_dataset_v1_statistics.json"
FINAL_STATS_CSV = DATA_FINAL_DIR / "idiomx_dataset_v1_source_distribution.csv"

# Copy files
shutil.copy2(STAGE6_CSV, FINAL_DATASET)
shutil.copy2(STATS_JSON, FINAL_STATS_JSON)
shutil.copy2(STATS_CSV, FINAL_STATS_CSV)

print("\nFinal dataset saved to:")
print(FINAL_DATASET)
print(FINAL_STATS_JSON)
print(FINAL_STATS_CSV)

Saved JSON stats: ..\data\processed\idiomx_dataset_v1_statistics.json
Saved source distribution CSV: ..\data\processed\idiomx_dataset_v1_source_distribution.csv

Summary:
Rows total: 12853
Unique idioms: 12847
Rows with meaning: 12853
Rows with example: 8891
Average idiom tokens: 3.23

Source distribution:
source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64
{'rows_total': 12853, 'unique_idioms': 12847, 'unique_meanings': 12525, 'rows_with_meaning': 12853, 'rows_with_example': 8891, 'rows_without_meaning': 0, 'rows_without_example': 3962, 'avg_idiom_tokens': 3.23, 'min_idiom_tokens': 2, 'max_idiom_tokens': 7, 'source_distribution': {'kaikki_wiktionary': 12838, 'wordnet': 15}, 'source_type_distribution': {'dictionary': 12838, 'lexical_database': 15}, 'idiom_confidence_distribution': {'high': 12838, 'medium': 15}}


,source,count
0,kaikki_wiktionary,12838
1,wordnet,15



Final dataset saved to:
..\data\final\idiomx_dataset_v1.csv
..\data\final\idiomx_dataset_v1_statistics.json
..\data\final\idiomx_dataset_v1_source_distribution.csv


# Findings and Conclusions

## Dataset Construction Overview

In this notebook, we constructed a comprehensive English idiom dataset by aggregating multiple publicly available lexical resources. The collection pipeline involved extracting idiomatic expressions from several heterogeneous sources, normalizing the data schema, merging the datasets, and applying rigorous filtering and deduplication procedures to ensure dataset quality.

The sources used in the dataset include:

* Wiktionary (via the Kaikki dataset)
* WordNet multi-word expressions

* * *

## Source Contribution Analysis

### Broad Dataset

* Wiktionary (Kaikki): **15,410 idioms**
* WordNet: **2,506 expressions**


### High-Precision Dataset

* Wiktionary (Kaikki): **12,838 idioms**
* WordNet: **15 idioms**

This distribution shows that **Wiktionary contributes the majority of verified idioms**, while WordNet mainly contributes general multi-word expressions that are filtered out during the precision stage.

* * *

## Dataset Quality Assessment

The final **IdiomX v1 dataset** contains:

* **12,853 unique idiomatic expressions**
* **High definition coverage (~99.7%)**
* Multiple trusted lexical sources
* Structured metadata including:

    * idiom
    * English meaning
    * example sentence
    * source
    * part-of-speech tags
    * confidence level

Compared to many published idiom datasets—which typically contain between **5,000 and 10,000 idioms**—IdiomX v1 represents a **large and high-quality idiom resource suitable for NLP research**.

* * *

## Final Dataset Output

The final dataset is saved as:

```
    data/final/idiomx_dataset_v1.csv
```

Additional metadata files include:

```
    idiomx_dataset_v1_statistics.json
    idiomx_dataset_v1_source_distribution.csv
```

These files provide reproducibility and transparency regarding the dataset composition.

* * *

## Conclusion

This work produced **IdiomX v1**, a curated English idiom dataset built through a multi-source data integration pipeline. By combining dictionary resources, lexical databases, and idiom repositories, and by applying systematic filtering and deduplication, we obtained a high-quality dataset of more than **12,000 idiomatic expressions**.

The resulting dataset provides a strong foundation for several NLP tasks, including:

* idiom detection
* idiom interpretation
* machine translation of idioms
* figurative language modeling
* idiom-aware language models

In future work, the dataset will be extended with **multilingual translations**, beginning with English-Arabic idiom pairs, enabling the training of specialized translation models capable of handling figurative language.

# Explore the Dataset

### imports and paths

In [25]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# =========================
# Paths
# =========================
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FINAL_DIR = DATA_DIR / "final"
PROCESSED_DIR = DATA_DIR / "processed"
ENRICHED_DIR = DATA_DIR / "enriched"
SAMPLES_DIR = DATA_DIR / "samples"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ENRICHED_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = FINAL_DIR / "idiomx_dataset_v1.csv"

print("Input file:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())

Input file: ..\data\final\idiomx_dataset_v1.csv
Exists: True


### load the dataset

In [26]:
df = pd.read_csv(INPUT_FILE)

print("Shape:", df.shape)
df.head(3)

Shape: (12853, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high,NaN
1,'fraid so,I am afraid so,NaN,kaikki_wiktionary,dictionary,phrase,slang,high,NaN
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high,NaN


### inspect columns and dtypes

In [27]:
print("Columns:\n")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

print("\n\nDtypes:\n")
print(df.dtypes)

print("\n\nMissing values per column:\n")
print(df.isna().sum().sort_values(ascending=False))

Columns:

01. idiom
02. meaning_en
03. example
04. source
05. source_type
06. pos
07. tags
08. idiom_confidence
09. source_url


Dtypes:

idiom                object
meaning_en           object
example              object
source               object
source_type          object
pos                  object
tags                 object
idiom_confidence     object
source_url          float64
dtype: object


Missing values per column:

source_url          12853
example              3962
tags                 1706
idiom                   0
meaning_en              0
source                  0
source_type             0
pos                     0
idiom_confidence        0
dtype: int64


### quick structural analysis

In [28]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_pct": ((df.isna().sum() / len(df)) * 100).round(2).values,
    "n_unique": df.nunique(dropna=True).values
}).sort_values(["null_pct", "n_unique"], ascending=[False, False])

summary

,column,dtype,non_null,null_count,null_pct,n_unique
8,source_url,float64,0,12853,100.00,0
2,example,object,8891,3962,30.83,8861
6,tags,object,11147,1706,13.27,1276
0,idiom,object,12853,0,0.00,12847
1,meaning_en,object,12853,0,0.00,12525
5,pos,object,12853,0,0.00,13
3,source,object,12853,0,0.00,2
4,source_type,object,12853,0,0.00,2
7,idiom_confidence,object,12853,0,0.00,2


### inspect important content distributions

In [29]:
# show source distribution if source exists
if "source" in df.columns:
    print("Source distribution:")
    display(df["source"].value_counts(dropna=False))

# show label distribution if label exists
if "label" in df.columns:
    print("\nLabel distribution:")
    display(df["label"].value_counts(dropna=False))

# generated example distribution if exists
if "is_generated_example" in df.columns:
    print("\nis_generated_example distribution:")
    display(df["is_generated_example"].value_counts(dropna=False))

# duplicate checks
for col in ["idiom_canonical", "idiom_surface", "example"]:
    if col in df.columns:
        print(f"\nDuplicate count based on {col}: {df.duplicated(subset=[col]).sum()}")

Source distribution:


source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64


Duplicate count based on example: 3991


### inspect row samples with text lengths

In [30]:
text_cols = [c for c in df.columns if df[c].dtype == "object"]

length_report = {}
for col in text_cols:
    length_report[col] = {
        "avg_len": round(df[col].fillna("").astype(str).str.len().mean(), 2),
        "max_len": int(df[col].fillna("").astype(str).str.len().max()),
        "min_len": int(df[col].fillna("").astype(str).str.len().min()),
    }

length_report_df = pd.DataFrame(length_report).T.sort_values("avg_len", ascending=False)
length_report_df

,avg_len,max_len,min_len
example,92.59,1336.0,0.0
meaning_en,66.67,375.0,4.0
source,16.99,17.0,7.0
idiom,15.87,51.0,4.0
tags,14.98,87.0,0.0
source_type,10.01,16.0,10.0
pos,4.93,11.0,1.0
idiom_confidence,4.00,6.0,4.0


### define the ideal publication schema

In [31]:
IDEAL_COLUMNS_ORDER = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "idiom_in_example",
    "idiom_canonical_meaning",
    "idiom_canonical_meaning_arabic",
    "label",
    "source",
    "is_generated_example",
    "record_origin",
    "license_source",
    "example_language",
    "meaning_language",
    "meaning_arabic_language",
]

### map current columns to ideal names

In [32]:
COLUMN_RENAME_MAP = {
    # keep if already correct
    "idiom_canonical": "idiom_canonical",
    "idiom_surface": "idiom_surface",
    "example": "example",
    "idiom_in_example": "idiom_in_example",
    "idiom_canonical_meaning": "idiom_canonical_meaning",
    "idiom_canonical_meaning_arabic": "idiom_canonical_meaning_arabic",
    "label": "label",
    "source": "source",
    "is_generated_example": "is_generated_example",

    # examples of possible alternate old names
    "canonical": "idiom_canonical",
    "surface": "idiom_surface",
    "meaning_en": "idiom_canonical_meaning",
    "meaning_ar": "idiom_canonical_meaning_arabic",
    "generated_example_flag": "is_generated_example",
}

In [33]:
df_pub = df.rename(columns=COLUMN_RENAME_MAP).copy()

print("Columns after rename:")
for col in df_pub.columns:
    print(col)

Columns after rename:
idiom
idiom_canonical_meaning
example
source
source_type
pos
tags
idiom_confidence
source_url


### add missing required columns

In [34]:
# stable unique id
if "idiom_id" not in df_pub.columns:
    df_pub.insert(0, "idiom_id", [f"idiomx_{i+1:06d}" for i in range(len(df_pub))])

# record origin
if "record_origin" not in df_pub.columns:
    if "is_generated_example" in df_pub.columns:
        df_pub["record_origin"] = np.where(
            df_pub["is_generated_example"].fillna(0).astype(int) == 1,
            "source_plus_llm",
            "source_only"
        )
    else:
        df_pub["record_origin"] = "source_only"

# source-based license
if "license_source" not in df_pub.columns:
    def infer_license(src):
        src = str(src).strip().lower()
        if src == "kaikki_wiktionary":
            return "wiktionary_cc_by_sa_4_0"
        elif src == "wordnet":
            return "wordnet"
        return "unknown"

    df_pub["license_source"] = df_pub["source"].apply(infer_license) if "source" in df_pub.columns else "unknown"

# language metadata
if "example_language" not in df_pub.columns:
    df_pub["example_language"] = "en"

if "meaning_language" not in df_pub.columns:
    df_pub["meaning_language"] = "en"

if "meaning_arabic_language" not in df_pub.columns:
    df_pub["meaning_arabic_language"] = "ar"

### standardize values and types

In [35]:
# strip whitespace from column names
df_pub.columns = [c.strip() for c in df_pub.columns]

# strip whitespace from object columns
for col in df_pub.select_dtypes(include="object").columns:
    df_pub[col] = df_pub[col].astype(str).str.strip()
    df_pub[col] = df_pub[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# standardize source names
if "source" in df_pub.columns:
    df_pub["source"] = df_pub["source"].replace({
        "kaikki": "kaikki_wiktionary",
        "wiktionary": "kaikki_wiktionary",
        "kaikki.org": "kaikki_wiktionary",
        "word_net": "wordnet",
        "WordNet": "wordnet",
    })

# standardize label
if "label" in df_pub.columns:
    df_pub["label"] = pd.to_numeric(df_pub["label"], errors="coerce").astype("Int64")

# standardize is_generated_example
if "is_generated_example" in df_pub.columns:
    df_pub["is_generated_example"] = (
        pd.to_numeric(df_pub["is_generated_example"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

# optional: lowercase normalization for selected metadata columns
for col in ["source", "record_origin", "license_source", "example_language", "meaning_language", "meaning_arabic_language"]:
    if col in df_pub.columns:
        df_pub[col] = df_pub[col].astype(str).str.strip().str.lower()

### quality checks

In [36]:
required_columns = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "idiom_canonical_meaning",
    "label",
    "source",
]

missing_required = [c for c in required_columns if c not in df_pub.columns]
print("Missing required columns:", missing_required)

print("\nNulls in required columns:")
for c in required_columns:
    if c in df_pub.columns:
        print(f"{c}: {df_pub[c].isna().sum()}")

print("\nDuplicate idiom_id:", df_pub["idiom_id"].duplicated().sum() if "idiom_id" in df_pub.columns else "N/A")

if all(c in df_pub.columns for c in ["idiom_canonical", "example", "label", "source"]):
    dup_core = df_pub.duplicated(subset=["idiom_canonical", "example", "label", "source"]).sum()
    print("Duplicate core rows:", dup_core)

Missing required columns: ['idiom_canonical', 'idiom_surface', 'label']

Nulls in required columns:
idiom_id: 0
example: 3962
idiom_canonical_meaning: 0
source: 0

Duplicate idiom_id: 0


### reorder columns to ideal order

In [37]:
existing_order = [c for c in IDEAL_COLUMNS_ORDER if c in df_pub.columns]
remaining_cols = [c for c in df_pub.columns if c not in existing_order]

df_pub = df_pub[existing_order + remaining_cols].copy()

print("Final ordered columns:")
print(df_pub.columns.tolist())
df_pub.head(3)

Final ordered columns:
['idiom_id', 'example', 'idiom_canonical_meaning', 'source', 'record_origin', 'license_source', 'example_language', 'meaning_language', 'meaning_arabic_language', 'idiom', 'source_type', 'pos', 'tags', 'idiom_confidence', 'source_url']


,idiom_id,example,idiom_canonical_meaning,source,record_origin,license_source,example_language,meaning_language,meaning_arabic_language,idiom,source_type,pos,tags,idiom_confidence,source_url
0,idiomx_000001,‘Look at that water! No wonder Duddle said he ...,Listen to you; listen to yourself; listen to it.,kaikki_wiktionary,source_only,wiktionary_cc_by_sa_4_0,en,en,ar,'ark at 'ee,dictionary,phrase,informal,high,NaN
1,idiomx_000002,<NA>,I am afraid so,kaikki_wiktionary,source_only,wiktionary_cc_by_sa_4_0,en,en,ar,'fraid so,dictionary,phrase,slang,high,NaN
2,idiomx_000003,"“Your new computer is super expensive!” “Yeah,...","Used either to end a discussion, or to imply t...",kaikki_wiktionary,source_only,wiktionary_cc_by_sa_4_0,en,en,ar,'nuff said,dictionary,phrase,informal,high,NaN


### create the reproducibility file before enrichment if available

In [38]:
pre_enrichment_cols = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "idiom_canonical_meaning",
    "source",
    "license_source",
]

existing_pre_cols = [c for c in pre_enrichment_cols if c in df_pub.columns]
df_pre = df_pub[existing_pre_cols].copy()

PRE_ENRICHMENT_PARQUET = PROCESSED_DIR / "idiomx_pre_enrichment.parquet"
PRE_ENRICHMENT_CSV = PROCESSED_DIR / "idiomx_pre_enrichment.csv"

df_pre.to_parquet(PRE_ENRICHMENT_PARQUET, index=False)
df_pre.to_csv(PRE_ENRICHMENT_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(PRE_ENRICHMENT_PARQUET)
print(PRE_ENRICHMENT_CSV)

Saved:
..\data\processed\idiomx_pre_enrichment.parquet
..\data\processed\idiomx_pre_enrichment.csv


### save the enriched full dataset

In [39]:
ENRICHED_FULL_PARQUET = ENRICHED_DIR / "idiomx_enriched_full.parquet"
ENRICHED_FULL_CSV = ENRICHED_DIR / "idiomx_enriched_full.csv"

df_pub.to_parquet(ENRICHED_FULL_PARQUET, index=False)
df_pub.to_csv(ENRICHED_FULL_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(ENRICHED_FULL_PARQUET)
print(ENRICHED_FULL_CSV)

Saved:
..\data\enriched\idiomx_enriched_full.parquet
..\data\enriched\idiomx_enriched_full.csv


### build the main public dataset

In [40]:
core_columns = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "idiom_in_example",
    "idiom_canonical_meaning",
    "idiom_canonical_meaning_arabic",
    "label",
    "source",
    "is_generated_example",
    "record_origin",
    "license_source",
    "example_language",
    "meaning_language",
    "meaning_arabic_language",
]

existing_core_columns = [c for c in core_columns if c in df_pub.columns]
df_core = df_pub[existing_core_columns].copy()

CORE_PARQUET = FINAL_DIR / "idiomx_core.parquet"
CORE_CSV = FINAL_DIR / "idiomx_core.csv"

df_core.to_parquet(CORE_PARQUET, index=False)
df_core.to_csv(CORE_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(CORE_PARQUET)
print(CORE_CSV)

Saved:
..\data\final\idiomx_core.parquet
..\data\final\idiomx_core.csv


### create the human-only subset

In [41]:
if "is_generated_example" in df_core.columns:
    df_human = df_core[df_core["is_generated_example"] == 0].copy()
else:
    df_human = df_core.copy()

HUMAN_PARQUET = FINAL_DIR / "idiomx_human_examples_only.parquet"
HUMAN_CSV = FINAL_DIR / "idiomx_human_examples_only.csv"

df_human.to_parquet(HUMAN_PARQUET, index=False)
df_human.to_csv(HUMAN_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(HUMAN_PARQUET)
print(HUMAN_CSV)
print("Human-only shape:", df_human.shape)

Saved:
..\data\final\idiomx_human_examples_only.parquet
..\data\final\idiomx_human_examples_only.csv
Human-only shape: (12853, 9)


### create a sample dataset

In [42]:
sample_n = min(1000, len(df_core))
df_sample = df_core.sample(sample_n, random_state=42).copy()

SAMPLE_PARQUET = SAMPLES_DIR / "idiomx_sample_1000.parquet"
SAMPLE_CSV = SAMPLES_DIR / "idiomx_sample_1000.csv"

df_sample.to_parquet(SAMPLE_PARQUET, index=False)
df_sample.to_csv(SAMPLE_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(SAMPLE_PARQUET)
print(SAMPLE_CSV)

Saved:
..\data\samples\idiomx_sample_1000.parquet
..\data\samples\idiomx_sample_1000.csv


### generate dataset statistics JSON

In [43]:
stats = {
    "dataset_name": "IdiomX",
    "version": "v1.0.0",
    "total_rows": int(len(df_core)),
    "total_columns": int(len(df_core.columns)),
    "unique_idioms_canonical": int(df_core["idiom_canonical"].nunique()) if "idiom_canonical" in df_core.columns else None,
    "source_distribution": df_core["source"].value_counts(dropna=False).to_dict() if "source" in df_core.columns else {},
    "label_distribution": df_core["label"].value_counts(dropna=False).to_dict() if "label" in df_core.columns else {},
    "generated_example_distribution": df_core["is_generated_example"].value_counts(dropna=False).to_dict() if "is_generated_example" in df_core.columns else {},
    "missing_values": df_core.isna().sum().to_dict(),
    "columns": df_core.columns.tolist(),
}

STATS_JSON = FINAL_DIR / "dataset_statistics.json"
with open(STATS_JSON, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print("Saved:", STATS_JSON)

Saved: ..\data\final\dataset_statistics.json


### save source distribution CSV

In [44]:
if "source" in df_core.columns:
    source_dist = (
        df_core["source"]
        .value_counts(dropna=False)
        .rename_axis("source")
        .reset_index(name="count")
    )
else:
    source_dist = pd.DataFrame(columns=["source", "count"])

SOURCE_DIST_CSV = FINAL_DIR / "source_distribution.csv"
source_dist.to_csv(SOURCE_DIST_CSV, index=False, encoding="utf-8-sig")

print("Saved:", SOURCE_DIST_CSV)
source_dist

Saved: ..\data\final\source_distribution.csv


,source,count
0,kaikki_wiktionary,12838
1,wordnet,15


### final verification report

In [45]:
print("=== FINAL FILES CREATED ===")
for path in [
    PRE_ENRICHMENT_PARQUET,
    PRE_ENRICHMENT_CSV,
    ENRICHED_FULL_PARQUET,
    ENRICHED_FULL_CSV,
    CORE_PARQUET,
    CORE_CSV,
    HUMAN_PARQUET,
    HUMAN_CSV,
    SAMPLE_PARQUET,
    SAMPLE_CSV,
    STATS_JSON,
    SOURCE_DIST_CSV,
]:
    print(path, "->", path.exists())

=== FINAL FILES CREATED ===
..\data\processed\idiomx_pre_enrichment.parquet -> True
..\data\processed\idiomx_pre_enrichment.csv -> True
..\data\enriched\idiomx_enriched_full.parquet -> True
..\data\enriched\idiomx_enriched_full.csv -> True
..\data\final\idiomx_core.parquet -> True
..\data\final\idiomx_core.csv -> True
..\data\final\idiomx_human_examples_only.parquet -> True
..\data\final\idiomx_human_examples_only.csv -> True
..\data\samples\idiomx_sample_1000.parquet -> True
..\data\samples\idiomx_sample_1000.csv -> True
..\data\final\dataset_statistics.json -> True
..\data\final\source_distribution.csv -> True


In [46]:
if "source" in df_pub.columns:
    display(df_pub.groupby("source").size().reset_index(name="count"))

if "source" in df_pub.columns and "idiom_canonical" in df_pub.columns:
    display(
        df_pub.groupby("source")["idiom_canonical"]
        .nunique()
        .reset_index(name="unique_idioms")
    )

,source,count
0,kaikki_wiktionary,12838
1,wordnet,15


### Drop useless column

In [48]:
df_clean = df.copy()

# drop useless column
df_clean = df_clean.drop(columns=["source_url"], errors="ignore")

print("Columns after drop:")
print(df_clean.columns.tolist())

Columns after drop:
['idiom', 'meaning_en', 'example', 'source', 'source_type', 'pos', 'tags', 'idiom_confidence']


### Rename to publication schema

In [49]:
df_clean = df_clean.rename(columns={
    "idiom": "idiom_canonical",
    "meaning_en": "idiom_canonical_meaning",
})

### Create missing required columns

In [50]:
# idiom_surface = same as canonical for now
df_clean["idiom_surface"] = df_clean["idiom_canonical"]

# create ID
df_clean.insert(
    0,
    "idiom_id",
    [f"idiomx_{i+1:06d}" for i in range(len(df_clean))]
)

### Fix idiom_confidence
Convert text → numeric score

In [51]:
confidence_map = {
    "high": 1.0,
    "medium": 0.7,
    "low": 0.4
}

df_clean["idiom_confidence_score"] = (
    df_clean["idiom_confidence"]
    .str.lower()
    .map(confidence_map)
)

# keep original if you want, or drop
df_clean = df_clean.drop(columns=["idiom_confidence"])

### Standardize text fields

In [52]:
# clean text columns
for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace({"": pd.NA, "nan": pd.NA})

# normalize source
df_clean["source"] = df_clean["source"].str.lower().str.strip()

### Add metadata columns

In [53]:
# license per source
def infer_license(src):
    if src == "kaikki_wiktionary":
        return "wiktionary_cc_by_sa_4_0"
    elif src == "wordnet":
        return "wordnet"
    return "unknown"

df_clean["license_source"] = df_clean["source"].apply(infer_license)

# record origin (no LLM yet)
df_clean["record_origin"] = "source_only"

# language metadata
df_clean["example_language"] = "en"
df_clean["meaning_language"] = "en"

### Reorder columns

In [54]:
final_columns = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "idiom_canonical_meaning",
    "source",
    "source_type",
    "pos",
    "tags",
    "idiom_confidence_score",
    "record_origin",
    "license_source",
    "example_language",
    "meaning_language",
]

df_clean = df_clean[final_columns]

### Validate

In [55]:
print("Final shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nSource distribution:")
print(df_clean["source"].value_counts())

Final shape: (12853, 14)

Missing values:
idiom_id                      0
idiom_canonical               0
idiom_surface                 0
example                    3962
idiom_canonical_meaning       0
source                        0
source_type                   0
pos                           0
tags                       1706
idiom_confidence_score        0
record_origin                 0
license_source                0
example_language              0
meaning_language              0
dtype: int64

Source distribution:
source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64


### re-enrichment save dataset to file

In [56]:
# Pre-enrichment
df_clean.to_parquet(
    PROCESSED_DIR / "idiomx_pre_enrichment.parquet",
    index=False
)

df_clean.to_csv(
    PROCESSED_DIR / "idiomx_pre_enrichment.csv",
    index=False,
    encoding="utf-8-sig"
)

In [63]:
df_clean.columns

Index(['idiom_id', 'idiom_canonical', 'idiom_surface', 'example',
       'idiom_canonical_meaning', 'source', 'source_type', 'pos', 'tags',
       'idiom_confidence_score', 'record_origin', 'license_source',
       'example_language', 'meaning_language'],
      dtype='object')

### save sample

In [57]:
df_clean.sample(1000).to_parquet(
    SAMPLES_DIR / "idiomx_sample_1000.parquet",
    index=False
)

In [58]:
df[df["source"] == "wordnet"].head(10)

,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
22,Ben Sira,an Apocryphal book mainly of maxims (resemblin...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
27,Book of Proverbs,an Old Testament book consisting of proverbs f...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
461,Wisdom of Jesus the Son of Sirach,an Apocryphal book mainly of maxims (resemblin...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
1508,blind alley,(figurative) a course of action that is unprod...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
2809,daisy chain,(figurative) a series of associated things or ...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
3741,figure of speech,language used in a figurative or nonliteral sense,NaN,wordnet,lexical_database,n,NaN,medium,NaN
4143,game plan,(figurative) a carefully thought out strategy ...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
4224,get off the ground,"get started or set in motion, used figuratively",NaN,wordnet,lexical_database,v,NaN,medium,NaN
5315,holy of holies,(figurative) something regarded as sacred or i...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
8312,peanut gallery,(figurative) people whose criticisms are regar...,NaN,wordnet,lexical_database,n,NaN,medium,NaN
